<link href="https://fonts.googleapis.com/css2?family=Oswald:wght@700&display=swap" rel="stylesheet">

<h1 style="
font-family: 'Oswald', sans-serif;
font-weight: 700;
font-style: italic;
font-size: 90px;
letter-spacing: 2px;
color: #E7C173;
text-shadow: 3px 3px 0 #333;
">
MACHINE LEARNING<br>IN INDUSTRY
</h1>

# Day 2 — Advanced: Modeling Diagnostics & Beyond

This notebook is for students who finish the **Core Day 2** notebook early or want deeper understanding.
It covers topics that go beyond the standard modeling pipeline: cross-validation mathematics,
metric pitfalls under class imbalance, covariate shift, adversarial validation, tabular foundation models,
Bayesian hyperparameter optimization, SHAP explanations, contrast model error analysis,
and ensemble stacking techniques used in competitive ML.

**Prerequisites:** You should have completed the Core Day 2 notebook (or be comfortable with
train/val/test splits, sklearn Pipelines, and basic evaluation metrics).

## Table of Contents

- [0. Setup & Data Loading](#setup)
- [1. The Mathematics Behind Cross-Validation](#cv-math)
- [2. AUPRC vs ROC-AUC Under Class Imbalance](#auprc)
- [3. When Test Performance Drops But It's Not Overfitting (Covariate Shift)](#covariate-shift)
- [4. Adversarial Validation](#adversarial)
- [5. Tabular Foundation Models](#foundation-models)
- [6. Bayesian Hyperparameter Optimization (Optuna)](#optuna)
- [7. SHAP: Beyond Feature Importance](#shap)
- [8. Contrast Model / Advanced Error Analysis](#contrast-model)
- [9. Ensemble Stacking & Blending](#stacking)

## <a id="setup"></a> Section 0 — Setup & Data Loading

Same compact setup as Core Day 2: load the Adult Income dataset, split, preprocess, and train
baseline models for use in the diagnostic sections that follow.

In [ ]:
import os
from pathlib import Path

_root = Path.cwd()
while _root != _root.parent and not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
print(f"Working directory: {Path.cwd()}")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, LeaveOneOut,
    GridSearchCV,
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve, brier_score_loss,
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
)

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)
plt.style.use("tableau-colorblind10")

import shap
import optuna

print("Environment ready.")
print(f"  shap:   {shap.__version__}")
print(f"  optuna: {optuna.__version__}")

In [ ]:
# ── Load & prepare data (same as Core) ──
adult = pd.read_csv("day1/generated/adult_income_issues.csv")

TARGET_COL = "class"
TARGET_BIN_COL = "target"
SPLIT_COL = "split"
ID_COLS = ["person_id"]
LEAKAGE_COLS = ["post_adjudication_risk_code"]
PROCESS_COLS = [
    "db_source_table", "db_etl_batch_id", "db_row_surrogate_key",
    "db_loaded_at_utc", "dataset_schema_version", "extract_country_code",
    "record_written_at", "dgp_regime",
]

adult[TARGET_BIN_COL] = (
    adult[TARGET_COL].astype(str).str.contains(">50", case=False, regex=False).astype(int)
)

# Dedup: keep latest record per person (same logic as Day 1)
adult = (adult.sort_values("record_written_at", ascending=False)
         .drop_duplicates("person_id", keep="first")
         .reset_index(drop=True))

def to_numeric_loose(series: pd.Series) -> pd.Series:
    cleaned = series.astype("string").str.strip().str.replace("h", "", regex=False)
    return pd.to_numeric(cleaned, errors="coerce")

# ── Split: use the dataset's split column, then stratified train/val ──
train_pool = adult.loc[adult[SPLIT_COL] == "train"].copy()
test_df = adult.loc[adult[SPLIT_COL] == "test"].copy()

# After dedup, each person_id has exactly one row → simple stratified split
train_df, val_df = train_test_split(
    train_pool, test_size=0.2, random_state=SEED,
    stratify=train_pool[TARGET_BIN_COL],
)

for name, frame in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:>5} rows={len(frame):5d}  target_rate={frame[TARGET_BIN_COL].mean():.4f}")

# ── Feature engineering (compact) ──
excluded = set(
    [TARGET_COL, TARGET_BIN_COL, SPLIT_COL]
    + ID_COLS + LEAKAGE_COLS + PROCESS_COLS
    + ["case_review_note", "constant_one"]
)
feature_cols = [c for c in adult.columns if c not in excluded]

numeric_cols, categorical_cols = [], []
for col in feature_cols:
    s = train_df[col]
    if s.dtype.kind in "biufc":
        numeric_cols.append(col)
        continue
    as_num = to_numeric_loose(s)
    if as_num.notna().mean() >= 0.9:
        numeric_cols.append(col)
    else:
        categorical_cols.append(col)

def build_features(df, numeric_cols, categorical_cols):
    X = pd.DataFrame(index=df.index)
    for c in numeric_cols:
        X[c] = to_numeric_loose(df[c])
    for c in categorical_cols:
        X[c] = df[c].astype(str).fillna("MISSING")
    return X

X_train_raw = build_features(train_df, numeric_cols, categorical_cols)
X_val_raw = build_features(val_df, numeric_cols, categorical_cols)
X_test_raw = build_features(test_df, numeric_cols, categorical_cols)

y_train = train_df[TARGET_BIN_COL].values
y_val = val_df[TARGET_BIN_COL].values
y_test = test_df[TARGET_BIN_COL].values

# ── Preprocessors ──
numeric_transformer_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
numeric_transformer_simple = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor_scaled = ColumnTransformer([
    ("num", numeric_transformer_scaled, numeric_cols),
    ("cat", categorical_transformer, categorical_cols),
])
preprocessor_tree = ColumnTransformer([
    ("num", numeric_transformer_simple, numeric_cols),
    ("cat", categorical_transformer, categorical_cols),
])

preprocessor_scaled.fit(X_train_raw)
X_train_sc = preprocessor_scaled.transform(X_train_raw)
X_val_sc = preprocessor_scaled.transform(X_val_raw)
X_test_sc = preprocessor_scaled.transform(X_test_raw)

preprocessor_tree.fit(X_train_raw)
X_train_tr = preprocessor_tree.transform(X_train_raw)
X_val_tr = preprocessor_tree.transform(X_val_raw)
X_test_tr = preprocessor_tree.transform(X_test_raw)

feature_names_ohe = preprocessor_tree.get_feature_names_out()
print(f"\nFeature matrices: train={X_train_tr.shape}, val={X_val_tr.shape}, test={X_test_tr.shape}")

In [ ]:
# ── Train baseline models for later sections ──
rf_baseline = RandomForestClassifier(
    n_estimators=300, max_depth=12, max_features="sqrt",
    min_samples_leaf=10, random_state=SEED, n_jobs=-1,
)
rf_baseline.fit(X_train_tr, y_train)
rf_val_auc = roc_auc_score(y_val, rf_baseline.predict_proba(X_val_tr)[:, 1])

gbm_baseline = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.1, max_depth=4,
    min_samples_leaf=20, random_state=SEED,
)
gbm_baseline.fit(X_train_tr, y_train)
gbm_val_auc = roc_auc_score(y_val, gbm_baseline.predict_proba(X_val_tr)[:, 1])

print(f"RandomForest   val AUC: {rf_val_auc:.4f}")
print(f"GBM (sklearn)  val AUC: {gbm_val_auc:.4f}")
print("\nBaseline models ready — these are used throughout the notebook.")

## <a id="cv-math"></a> Section 1 — The Mathematics Behind Cross-Validation & Why Stratified Sampling Helps

Cross-validation provides an estimate of the **generalization error** — the expected loss on unseen data
drawn from the same distribution as the training data.

### Bias-Variance Decomposition of the CV Error

The CV estimator has two sources of error:

- **Bias:** With K folds, each training set has (K-1)/K of the full data. Fewer training examples →
  the model underfits slightly → the CV estimate is *pessimistic* (biased upward for error, downward for AUC).
  Larger K reduces this bias.

- **Variance:** Each fold's estimate is noisy. With large K, the training folds overlap heavily,
  making the estimates *correlated* — this increases the variance of the average.
  The extreme case (K=n, LOOCV) has maximum overlap and can have very high variance.

### The K=5 or K=10 Sweet Spot

Hastie, Tibshirani & Friedman (*Elements of Statistical Learning*, §7.10) recommend K=5 or K=10 as
a practical compromise. Empirically, K=10 has good bias-variance balance for most problems.

| K | Training fraction | Bias | Variance |
|---|-------------------|------|----------|
| 2 | 50% | High (pessimistic) | Low |
| 5 | 80% | Moderate | Moderate |
| 10 | 90% | Low | Moderate |
| n (LOOCV) | ~100% | Very low | High (correlated folds) |

### Why Stratification?

For imbalanced classification, a random K-fold split can produce folds where the positive-class rate
varies wildly. Stratified K-fold preserves the target distribution in each fold, which:
1. Reduces variance of the metric estimate
2. Ensures every fold has enough positives for meaningful precision/recall computation

📚 [Hastie et al., ESL §7.10](https://hastie.su.domains/ElemStatLearn/) · [sklearn StratifiedKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)

In [ ]:
# ── Demonstration: K=2 (high bias) vs K=10 vs LOOCV (high variance) ──
# We use a small subsample so LOOCV is tractable
from sklearn.model_selection import LeaveOneOut

rng = np.random.RandomState(SEED)
subsample_idx = rng.choice(len(X_train_sc), size=200, replace=False)
X_sub = X_train_sc[subsample_idx]
y_sub = y_train[subsample_idx]

model_cv = LogisticRegression(C=1.0, max_iter=500, random_state=SEED)

results_k = {}
for k_val in [2, 3, 5, 10]:
    skf = StratifiedKFold(k_val, shuffle=True, random_state=SEED)
    scores = cross_val_score(model_cv, X_sub, y_sub, cv=skf, scoring="roc_auc")
    results_k[f"K={k_val}"] = {"mean": scores.mean(), "std": scores.std(), "scores": scores}

# LOOCV (takes a moment on 200 samples)
loo = LeaveOneOut()
loo_scores = cross_val_score(model_cv, X_sub, y_sub, cv=loo, scoring="roc_auc")
results_k["LOOCV"] = {"mean": loo_scores.mean(), "std": loo_scores.std(), "scores": loo_scores}

print(f"{'Method':<10} {'Mean AUC':>10} {'Std':>10}")
print("-" * 32)
for name, r in results_k.items():
    print(f"{name:<10} {r['mean']:>10.4f} {r['std']:>10.4f}")

print("\nK=2 has high bias (pessimistic estimate from training on only 50% of data).")
print("LOOCV has low bias but can have high variance when samples are correlated.")

In [ ]:
# ── Simulation: random splits vs stratified splits on imbalanced data ──
# Create a more imbalanced version (keep only 10% of positives)
rng = np.random.RandomState(SEED)

pos_idx = np.where(y_train == 1)[0]
neg_idx = np.where(y_train == 0)[0]
keep_pos = rng.choice(pos_idx, size=int(len(pos_idx) * 0.3), replace=False)
imb_idx = np.concatenate([neg_idx, keep_pos])
rng.shuffle(imb_idx)

X_imb = X_train_tr[imb_idx]
y_imb = y_train[imb_idx]
true_rate = y_imb.mean()
print(f"Imbalanced dataset: n={len(y_imb)}, positive rate={true_rate:.4f}")

# 100 random 5-fold splits vs 100 stratified 5-fold splits
n_reps = 100
random_rates = []
stratified_rates = []

for i in range(n_reps):
    # Random KFold
    from sklearn.model_selection import KFold
    kf = KFold(5, shuffle=True, random_state=i)
    for _, test_idx in kf.split(X_imb):
        random_rates.append(y_imb[test_idx].mean())

    # Stratified KFold
    skf = StratifiedKFold(5, shuffle=True, random_state=i)
    for _, test_idx in skf.split(X_imb, y_imb):
        stratified_rates.append(y_imb[test_idx].mean())

random_rates = np.array(random_rates)
stratified_rates = np.array(stratified_rates)

print(f"\nRandom splits:     mean rate={random_rates.mean():.4f}, std={random_rates.std():.4f}")
print(f"Stratified splits: mean rate={stratified_rates.mean():.4f}, std={stratified_rates.std():.4f}")
print(f"Variance reduction: {random_rates.std() / stratified_rates.std():.1f}x tighter with stratification")

In [ ]:
# ── Plot: distribution of positive-class rate per fold ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(random_rates, bins=30, alpha=0.7, color="steelblue", edgecolor="white", label="Random KFold")
axes[0].axvline(true_rate, color="red", linestyle="--", linewidth=2, label=f"True rate ({true_rate:.3f})")
axes[0].set_xlabel("Positive-class rate in fold")
axes[0].set_ylabel("Count")
axes[0].set_title("Random KFold — Fold Target Rates")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(stratified_rates, bins=30, alpha=0.7, color="darkorange", edgecolor="white", label="Stratified KFold")
axes[1].axvline(true_rate, color="red", linestyle="--", linewidth=2, label=f"True rate ({true_rate:.3f})")
axes[1].set_xlabel("Positive-class rate in fold")
axes[1].set_ylabel("Count")
axes[1].set_title("Stratified KFold — Fold Target Rates")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Match x-axis for fair comparison
xmin = min(random_rates.min(), stratified_rates.min()) - 0.01
xmax = max(random_rates.max(), stratified_rates.max()) + 0.01
axes[0].set_xlim(xmin, xmax)
axes[1].set_xlim(xmin, xmax)

plt.tight_layout()
plt.show()

print("Stratified sampling produces much tighter control over class balance per fold.")
print("This translates to lower variance in CV metric estimates — especially for precision and recall.")

> **Do It Yourself**
>
> Repeat the simulation above using `GroupKFold` with `person_id` as the group.
> How does the positive-class rate distribution compare to stratified? Can you combine
> group-awareness with stratification? (Hint: look at `StratifiedGroupKFold` in sklearn ≥ 1.0.)

## <a id="auprc"></a> Section 2 — AUPRC vs ROC-AUC Under Class Imbalance / Changing Prevalence

ROC-AUC measures the probability that a randomly chosen positive is scored higher than a randomly
chosen negative. It is **independent of prevalence** — which is both its strength and its weakness.

**PR-AUC** (Average Precision) measures the area under the precision-recall curve. Because precision
is defined as TP/(TP+FP), it is directly affected by the number of false positives — which grows
when the positive class is rare.

**Key insight:** A model with ROC-AUC = 0.90 might look great, but if only 1% of your data is
positive, the same model could have PR-AUC = 0.20. The ROC-AUC hides the fact that most
"positive" predictions are false positives.

📚 [Saito & Rehmsmeier (2015), "The Precision-Recall Plot Is More Informative than the ROC Plot"](https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0118432) · [sklearn average_precision_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html)

In [ ]:
# ── Synthetic experiment: same model, varying prevalence ──
from sklearn.datasets import make_classification

prevalence_levels = [0.01, 0.05, 0.10, 0.20, 0.50]
results = []

for prev in prevalence_levels:
    n_total = 5000
    n_pos = int(n_total * prev)
    n_neg = n_total - n_pos

    # Create a dataset with controlled prevalence but same separability
    X_syn, y_syn = make_classification(
        n_samples=n_total, n_features=20, n_informative=10,
        weights=[1 - prev, prev], flip_y=0.05,
        random_state=SEED, class_sep=1.0,
    )

    X_syn_train, X_syn_test, y_syn_train, y_syn_test = train_test_split(
        X_syn, y_syn, test_size=0.3, random_state=SEED, stratify=y_syn,
    )

    clf = GradientBoostingClassifier(
        n_estimators=100, max_depth=3, random_state=SEED,
    )
    clf.fit(X_syn_train, y_syn_train)
    y_prob_syn = clf.predict_proba(X_syn_test)[:, 1]

    roc = roc_auc_score(y_syn_test, y_prob_syn)
    pr = average_precision_score(y_syn_test, y_prob_syn)

    results.append({
        "prevalence": prev,
        "n_pos_test": y_syn_test.sum(),
        "roc_auc": roc,
        "pr_auc": pr,
    })
    print(f"Prevalence={prev:.0%}: ROC-AUC={roc:.4f}, PR-AUC={pr:.4f} (n_pos_test={y_syn_test.sum()})")

results_df = pd.DataFrame(results)

In [ ]:
# ── Visualization: ROC-AUC stays flat, PR-AUC drops ──
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(results_df["prevalence"], results_df["roc_auc"], "o-", markersize=8,
        linewidth=2, label="ROC-AUC", color="steelblue")
ax.plot(results_df["prevalence"], results_df["pr_auc"], "s-", markersize=8,
        linewidth=2, label="PR-AUC (Average Precision)", color="darkorange")

ax.set_xlabel("Prevalence (positive class rate)", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("ROC-AUC vs PR-AUC Under Varying Prevalence", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

# Annotate the key insight
ax.annotate(
    "ROC-AUC barely changes\nwhile PR-AUC drops dramatically",
    xy=(0.05, results_df.loc[results_df["prevalence"] == 0.05, "pr_auc"].values[0]),
    xytext=(0.20, 0.30),
    arrowprops=dict(arrowstyle="->", color="red"),
    fontsize=10, color="red",
)

plt.tight_layout()
plt.show()

print("Lesson: when the positive class is rare, ROC-AUC can be misleadingly optimistic.")
print("Always report PR-AUC alongside ROC-AUC for imbalanced problems.")

> **Do It Yourself**
>
> Take the Adult Income dataset and downsample the positive class to 5% prevalence.
> Train the same GBM model on the original and downsampled versions.
> Compare ROC-AUC and PR-AUC — does the pattern hold on real data?

## <a id="covariate-shift"></a> Section 3 — When Test Performance Drops But It's Not Overfitting (Covariate Shift)

**Covariate shift** occurs when the distribution of input features P(X) changes between training
and deployment (test), even though the conditional distribution P(Y|X) stays the same.

This is **not overfitting**: your model generalizes well within the training distribution — it just
encounters different inputs at test time.

**Examples in industry:**
- A credit scoring model trained on applicants from 2019, deployed during COVID (2020) — income distributions shifted
- A recommendation system trained on desktop users, deployed to mobile — feature distributions differ
- A medical model trained at Hospital A, deployed at Hospital B — patient demographics differ

The diagnostic signature:
- Train AUC ≈ Val AUC (both from distribution A)
- Test AUC drops (distribution B)
- **No train-val gap** → not overfitting!

📚 [Sugiyama et al. (2007), "Covariate Shift Adaptation"](https://mitpress.mit.edu/9780262170055/dataset-shift-in-machine-learning/) · Quinonero-Candela et al., *Dataset Shift in Machine Learning*

In [ ]:
# ── Simulate covariate shift by modifying test distribution ──
# We'll shift the 'age' feature (or first numeric feature) in the test set

# Identify a numeric feature to shift
shift_col_idx = 0  # first numeric column in the preprocessed matrix
shift_col_name = numeric_cols[0] if numeric_cols else "feature_0"

print(f"Shifting feature: '{shift_col_name}' (index {shift_col_idx})")
print(f"Original test mean: {X_test_tr[:, shift_col_idx].mean():.2f}")
print(f"Original train mean: {X_train_tr[:, shift_col_idx].mean():.2f}")

# Create shifted test set: add 2 standard deviations to the feature
X_test_shifted = X_test_tr.copy()
shift_amount = X_train_tr[:, shift_col_idx].std() * 2
X_test_shifted[:, shift_col_idx] += shift_amount

print(f"\nShifted test mean: {X_test_shifted[:, shift_col_idx].mean():.2f}")
print(f"Shift amount: +{shift_amount:.2f} (2 std devs)")

# Evaluate
train_auc = roc_auc_score(y_train, gbm_baseline.predict_proba(X_train_tr)[:, 1])
val_auc = roc_auc_score(y_val, gbm_baseline.predict_proba(X_val_tr)[:, 1])
test_orig_auc = roc_auc_score(y_test, gbm_baseline.predict_proba(X_test_tr)[:, 1])
test_shift_auc = roc_auc_score(y_test, gbm_baseline.predict_proba(X_test_shifted)[:, 1])

print(f"\n{'Set':<20} {'AUC':>8}")
print("-" * 30)
print(f"{'Train':<20} {train_auc:>8.4f}")
print(f"{'Validation':<20} {val_auc:>8.4f}")
print(f"{'Test (original)':<20} {test_orig_auc:>8.4f}")
print(f"{'Test (shifted)':<20} {test_shift_auc:>8.4f}")
print(f"\nTrain ≈ Val (no overfitting), but shifted test drops → covariate shift!")

In [ ]:
# ── Feature distribution comparison: train vs shifted test ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of the shifted feature
axes[0].hist(X_train_tr[:, shift_col_idx], bins=40, alpha=0.6, color="steelblue",
             label="Train", density=True, edgecolor="white")
axes[0].hist(X_test_shifted[:, shift_col_idx], bins=40, alpha=0.6, color="darkorange",
             label="Test (shifted)", density=True, edgecolor="white")
axes[0].axvline(X_train_tr[:, shift_col_idx].mean(), color="steelblue", linestyle="--", linewidth=2)
axes[0].axvline(X_test_shifted[:, shift_col_idx].mean(), color="darkorange", linestyle="--", linewidth=2)
axes[0].set_title(f"Feature '{shift_col_name}' — Train vs Shifted Test")
axes[0].set_xlabel(shift_col_name)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# AUC comparison bar chart
labels = ["Train", "Val", "Test\n(original)", "Test\n(shifted)"]
aucs = [train_auc, val_auc, test_orig_auc, test_shift_auc]
colors = ["steelblue", "steelblue", "steelblue", "darkorange"]
axes[1].bar(labels, aucs, color=colors, alpha=0.7, edgecolor="black")
axes[1].set_ylabel("AUC")
axes[1].set_title("AUC Across Sets — Covariate Shift Signature")
axes[1].set_ylim(0.5, 1.0)
axes[1].grid(True, alpha=0.3, axis="y")

for i, v in enumerate(aucs):
    axes[1].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

print("In the real world, you don't get to 'shift' test data yourself — the shift happens naturally.")
print("The next section shows how to DETECT it: adversarial validation.")

> **Do It Yourself**
>
> Instead of shifting a single feature, try shifting 3 numeric features simultaneously.
> How much does AUC degrade? What if you shift features the model doesn't care about
> (low permutation importance)? Does the AUC still drop?

## <a id="adversarial"></a> Section 4 — Adversarial Validation

**The recipe:**
1. Label all training rows as 0, all test rows as 1
2. Train a classifier to distinguish train from test
3. If the AUC >> 0.5, the distributions are different → covariate shift exists
4. Inspect feature importances to see which features drive the distinction

This technique is widely used in Kaggle competitions and industry to diagnose distribution mismatch
before a model even sees the target variable.

📚 [Adversarial Validation (Kaggle blog)](https://www.kaggle.com/code/carlmcbrideellis/what-is-adversarial-validation) · Pan & Yang (2010), 'A Survey on Transfer Learning'

In [ ]:
# ── Adversarial validation: can a model distinguish train from test? ──

# Combine train and test feature matrices
X_adv = np.vstack([X_train_tr, X_test_tr])
y_adv = np.concatenate([np.zeros(len(X_train_tr)), np.ones(len(X_test_tr))])

print(f"Adversarial dataset: {len(X_adv)} rows ({len(X_train_tr)} train + {len(X_test_tr)} test)")

# Train a classifier to distinguish train vs test
adv_clf = GradientBoostingClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.1, random_state=SEED,
)
adv_cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
adv_scores = cross_val_score(adv_clf, X_adv, y_adv, cv=adv_cv, scoring="roc_auc")

print(f"\nAdversarial validation AUC: {adv_scores.mean():.4f} ± {adv_scores.std():.4f}")

if adv_scores.mean() > 0.55:
    print("⚠ AUC > 0.55: train and test distributions are distinguishable!")
    print("  Some features differ between train and test — investigate below.")
else:
    print("AUC ≈ 0.50: train and test look similar — no obvious covariate shift.")

In [ ]:
# ── Which features drive the train/test distinction? ──
adv_clf.fit(X_adv, y_adv)
adv_importances = pd.DataFrame({
    "feature": feature_names_ohe,
    "importance": adv_clf.feature_importances_,
}).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
top_adv = adv_importances.head(15)
ax.barh(top_adv["feature"], top_adv["importance"], color="coral", alpha=0.7)
ax.set_xlabel("Feature Importance (adversarial model)")
ax.set_title("Adversarial Validation — Features That Distinguish Train from Test")
ax.invert_yaxis()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Top features by adversarial importance are the ones that differ most between train and test.")
print("These are candidates for:")
print("  1. Removal (if they're artifacts, e.g., data collection differences)")
print("  2. Alignment (if they represent genuine shift that should be handled)")
print("  3. Monitoring (track these features in production for drift detection)")

> **Do It Yourself**
>
> Run adversarial validation on the shifted test set from §3. The AUC should be much higher.
> Which feature does the adversarial model rank as most important? Does it match the feature you shifted?

## <a id="foundation-models"></a> Section 5 — Tabular Foundation Models (~15 min intro)

### What Are Tabular Foundation Models?

Just as GPT-4 is pretrained on text and can be applied to new tasks, **tabular foundation models**
are pretrained on many tabular datasets and can be applied to new tables without training.

Key examples:
- **TabPFN** (Hollmann et al., 2023): A Prior-Data Fitted Network that performs Bayesian inference
  in a single forward pass. Works on datasets up to ~10K rows, ~100 features.
- **TabICL** (Ye et al., 2025): In-context learning for tabular data, more scalable than TabPFN.

### How They Work

These models are trained on millions of synthetic datasets with known generative processes.
At inference time, they receive the training data + a test row as "context" and predict directly —
no gradient updates needed. It's like few-shot prompting, but for tables.

### Current State (2025)

| Aspect | Status |
|--------|--------|
| Small datasets (<10K rows) | Competitive with GBM |
| Large datasets (>50K rows) | Still behind well-tuned GBM |
| Feature engineering | Less needed (model learns representations) |
| Production readiness | Early — active research area |
| Speed | Fast inference, no training |

**Where the field is going:** these models are improving rapidly. In 2-3 years, they may become
the default starting point for tabular tasks, similar to how pretrained transformers became the
default for NLP.

📚 [TabPFN paper](https://arxiv.org/abs/2207.01848) · [TabPFN GitHub](https://github.com/automl/TabPFN) · [TabICL paper](https://arxiv.org/abs/2501.02349)

In [ ]:
# ── Demo: TabPFN API pattern (commented out — optional install) ──
# Install: pip install tabpfn
#
# from tabpfn import TabPFNClassifier
#
# tabpfn = TabPFNClassifier(device="cpu", N_ensemble_configurations=16)
#
# # TabPFN takes raw train data and predicts on test — no .fit() / .predict() separation
# # (internally it processes train+test in a single forward pass)
# tabpfn.fit(X_train_tr[:1000], y_train[:1000])  # subset — TabPFN has row limits
# y_prob_tabpfn = tabpfn.predict_proba(X_val_tr)[:, 1]
# tabpfn_auc = roc_auc_score(y_val, y_prob_tabpfn)
# print(f"TabPFN val AUC: {tabpfn_auc:.4f}")

# ── Demo: TabICL API pattern (commented out — optional install) ──
# Install: pip install tabicl
#
# from tabicl import TabICLClassifier
#
# tabicl = TabICLClassifier()
# tabicl.fit(X_train_tr, y_train)
# y_prob_tabicl = tabicl.predict_proba(X_val_tr)[:, 1]
# tabicl_auc = roc_auc_score(y_val, y_prob_tabicl)
# print(f"TabICL val AUC: {tabicl_auc:.4f}")

print("TabPFN and TabICL are commented out (optional installs).")
print("Uncomment and install to try them — they offer a glimpse of where the field is heading.")
print()
print("Key idea: these models require NO training on your dataset.")
print("They predict by treating your train set as context — like few-shot prompting for tables.")

> **Do It Yourself**
>
> If you install TabPFN, try it on a subset of the Adult dataset (first 1000 rows).
> Compare its AUC to the GBM baseline. How does it do without any hyperparameter tuning?

## <a id="optuna"></a> Section 6 — Bayesian Hyperparameter Optimization (Optuna)

### Grid Search vs Random Search vs Bayesian Optimization

| Method | Search Strategy | Efficiency | Scales to |
|--------|----------------|------------|-----------|
| Grid Search | Exhaustive | Low (exponential in # params) | 2-3 params |
| Random Search | Random sampling | Better (Bergstra & Bengio, 2012) | 5-10 params |
| Bayesian (Optuna) | Model-based: learns which regions of the space are promising | Best | 10+ params |

**Optuna** uses Tree-structured Parzen Estimators (TPE) to build a probabilistic model of
the objective function. Each trial informs the next, concentrating search in promising regions.

Key Optuna features:
- **Pruning:** stop unpromising trials early (saves time)
- **Multi-objective:** optimize multiple metrics simultaneously
- **Visualization:** built-in plots for optimization history, parameter importance

📚 [Optuna docs](https://optuna.readthedocs.io/) · [Akiba et al. (2019), "Optuna: A Next-generation Hyperparameter Optimization Framework"](https://arxiv.org/abs/1907.10902)

In [ ]:
# ── Optuna study: tune GBM hyperparameters ──
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 50),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
    }

    model = GradientBoostingClassifier(random_state=SEED, **params)
    skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
    scores = cross_val_score(model, X_train_tr, y_train, cv=skf, scoring="roc_auc", n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction="maximize", study_name="gbm_tuning")
study.optimize(objective, n_trials=40, show_progress_bar=True)

print(f"\nBest trial:")
print(f"  Value (CV AUC): {study.best_trial.value:.4f}")
print(f"  Params: {study.best_trial.params}")

# Train final model with best params
best_gbm = GradientBoostingClassifier(random_state=SEED, **study.best_trial.params)
best_gbm.fit(X_train_tr, y_train)
optuna_val_auc = roc_auc_score(y_val, best_gbm.predict_proba(X_val_tr)[:, 1])
print(f"  Val AUC (hold-out): {optuna_val_auc:.4f}")
print(f"  Baseline GBM val AUC: {gbm_val_auc:.4f}")
print(f"  Improvement: {optuna_val_auc - gbm_val_auc:+.4f}")

In [ ]:
# ── Optuna visualizations ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Optimization history
trials = study.trials
trial_numbers = [t.number for t in trials]
trial_values = [t.value for t in trials]
best_so_far = np.maximum.accumulate(trial_values)

axes[0].scatter(trial_numbers, trial_values, alpha=0.6, s=30, label="Trial AUC")
axes[0].plot(trial_numbers, best_so_far, "r-", linewidth=2, label="Best so far")
axes[0].set_xlabel("Trial Number")
axes[0].set_ylabel("CV AUC")
axes[0].set_title("Optuna Optimization History")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Parameter importance (manual approximation)
param_names = list(study.best_trial.params.keys())
param_importance = {}
for pname in param_names:
    vals = [t.params.get(pname, None) for t in trials if t.value is not None]
    scores = [t.value for t in trials if t.value is not None]
    if all(isinstance(v, (int, float)) for v in vals if v is not None):
        valid = [(v, s) for v, s in zip(vals, scores) if v is not None]
        if len(valid) > 5:
            corr = abs(np.corrcoef([v[0] for v in valid], [v[1] for v in valid])[0, 1])
            param_importance[pname] = corr if np.isfinite(corr) else 0
        else:
            param_importance[pname] = 0
    else:
        param_importance[pname] = 0

sorted_params = sorted(param_importance.items(), key=lambda x: x[1], reverse=True)
axes[1].barh([p[0] for p in sorted_params], [p[1] for p in sorted_params],
             color="steelblue", alpha=0.7)
axes[1].set_xlabel("|Correlation| with objective")
axes[1].set_title("Parameter Importance (correlation-based)")
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Compare Optuna result vs GridSearchCV ──
# Quick GridSearchCV for comparison
param_grid_gs = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.05, 0.1],
}
gs = GridSearchCV(
    GradientBoostingClassifier(random_state=SEED),
    param_grid_gs,
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    scoring="roc_auc",
    n_jobs=-1,
)
gs.fit(X_train_tr, y_train)
gs_val_auc = roc_auc_score(y_val, gs.predict_proba(X_val_tr)[:, 1])

print(f"{'Method':<25} {'CV AUC':>10} {'Val AUC':>10} {'Trials':>8}")
print("-" * 55)
print(f"{'GridSearchCV':<25} {gs.best_score_:>10.4f} {gs_val_auc:>10.4f} {len(param_grid_gs['n_estimators']) * len(param_grid_gs['max_depth']) * len(param_grid_gs['learning_rate']):>8}")
print(f"{'Optuna (40 trials)':<25} {study.best_trial.value:>10.4f} {optuna_val_auc:>10.4f} {len(study.trials):>8}")
print(f"{'Baseline (manual)':<25} {'—':>10} {gbm_val_auc:>10.4f} {'1':>8}")
print("\nOptuna explores a much larger space with the same or fewer evaluations.")

> **Do It Yourself**
>
> Extend the Optuna study to also tune `max_features` (fraction of features per split)
> and run 60 trials. Does it find a better configuration? Try adding early stopping
> by using `optuna.integration.OptunaSearchCV` with a sklearn Pipeline.

## <a id="shap"></a> Section 7 — SHAP: Beyond Feature Importance

**SHAP** (SHapley Additive exPlanations) provides a principled way to decompose each prediction
into feature contributions, based on Shapley values from cooperative game theory.

Key advantages over permutation importance:
- **Local explanations:** understand *why* a specific prediction was made
- **Signed:** tells you whether a feature pushes the prediction up or down
- **Additive:** contributions sum to the difference between prediction and baseline
- **Consistent:** if a feature's true contribution increases, its SHAP value increases

For tree-based models, `TreeExplainer` computes exact Shapley values in polynomial time
(Lundberg et al., 2020) — making it practical even for large datasets.

📚 [SHAP docs](https://shap.readthedocs.io/) · [Lundberg & Lee (2017), "A Unified Approach to Interpreting Model Predictions"](https://arxiv.org/abs/1705.07874)

In [ ]:
# ── SHAP values for the Random Forest baseline ──
explainer = shap.TreeExplainer(rf_baseline)
# Use a subsample for speed
sample_idx = np.random.RandomState(SEED).choice(len(X_val_tr), size=min(500, len(X_val_tr)), replace=False)
X_shap = X_val_tr[sample_idx]
shap_values_raw = explainer.shap_values(X_shap)

# For binary classification, TreeExplainer may return:
# - list of [class_0_array, class_1_array] (older shap versions)
# - 3D array of shape (n_samples, n_features, n_classes)
# - 2D array (n_samples, n_features)
if isinstance(shap_values_raw, list):
    shap_vals = shap_values_raw[1]
elif shap_values_raw.ndim == 3:
    shap_vals = shap_values_raw[:, :, 1]
else:
    shap_vals = shap_values_raw

# Ensure 2D and matching columns
if shap_vals.ndim != 2 or shap_vals.shape[1] != X_shap.shape[1]:
    print(f"WARNING: SHAP shape mismatch. shap_vals={shap_vals.shape}, X_shap={X_shap.shape}")
    print("Attempting to reshape...")
    if shap_vals.ndim == 3:
        shap_vals = shap_vals[:, :, 1] if shap_vals.shape[2] > 1 else shap_vals[:, :, 0]

print(f"SHAP values shape: {shap_vals.shape}")
print(f"X_shap shape: {X_shap.shape}")

# Summary plot (beeswarm)
print("--- Beeswarm Plot (global feature importance with direction) ---")
shap.summary_plot(shap_vals, X_shap, feature_names=feature_names_ohe, show=True, max_display=15)

In [ ]:
# ── SHAP dependence plots for top features ──
# Get top 4 features by mean absolute SHAP value
mean_abs_shap = np.abs(shap_vals).mean(axis=0)
top_4_idx = np.argsort(mean_abs_shap)[-4:][::-1]
top_4_names = [feature_names_ohe[i] for i in top_4_idx]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, idx, name in zip(axes.ravel(), top_4_idx, top_4_names):
    ax.scatter(X_shap[:, idx], shap_vals[:, idx], alpha=0.3, s=5)
    ax.set_xlabel(name)
    ax.set_ylabel(f"SHAP value for {name}")
    ax.grid(True, alpha=0.3)
plt.suptitle("SHAP Dependence Plots — Top 4 Features", fontsize=14)
plt.tight_layout()
plt.show()

print("Dependence plots show the relationship between feature value and SHAP value.")
print("Positive SHAP = pushes prediction toward positive class.")

In [ ]:
# ── Compare SHAP ranking vs permutation importance ranking ──
# SHAP-based ranking
shap_importance = pd.DataFrame({
    "feature": feature_names_ohe,
    "shap_mean_abs": np.abs(shap_vals).mean(axis=0),
}).sort_values("shap_mean_abs", ascending=False).reset_index(drop=True)
shap_importance["shap_rank"] = range(1, len(shap_importance) + 1)

# Permutation importance ranking
perm_imp = permutation_importance(
    rf_baseline, X_val_tr, y_val,
    n_repeats=10, random_state=SEED, scoring="roc_auc", n_jobs=-1,
)
perm_importance = pd.DataFrame({
    "feature": feature_names_ohe,
    "perm_mean": perm_imp.importances_mean,
}).sort_values("perm_mean", ascending=False).reset_index(drop=True)
perm_importance["perm_rank"] = range(1, len(perm_importance) + 1)

# Merge and compare top 15
comparison = shap_importance.merge(perm_importance, on="feature")
comparison = comparison.sort_values("shap_rank").head(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(comparison["feature"], comparison["shap_mean_abs"], color="steelblue", alpha=0.7)
axes[0].set_title("SHAP Importance (mean |SHAP value|)")
axes[0].invert_yaxis()
axes[0].grid(True, alpha=0.3)

axes[1].barh(comparison["feature"], comparison["perm_mean"], color="darkorange", alpha=0.7)
axes[1].set_title("Permutation Importance (AUC drop)")
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Rankings may differ because:")
print("  - SHAP captures direction and magnitude; permutation importance only captures magnitude")
print("  - Permutation importance can underestimate correlated features")
print("  - SHAP is model-specific; permutation importance is model-agnostic")

> **Do It Yourself**
>
> Compute SHAP values for the GBM model instead of Random Forest.
> Do the SHAP rankings agree between the two models?
> Try a force plot for a single prediction: `shap.force_plot(explainer.expected_value[1], shap_vals[0], ...)` —
> which features pushed that specific individual toward >50K?

## <a id="contrast-model"></a> Section 8 — Contrast Model / Advanced Error Analysis

### The Idea

Your model makes errors. But *where* and *why* it makes errors is often more informative than
aggregate error metrics. A **contrast model** (also called an error model or meta-model) treats
misclassification as the target:

1. Generate predictions from your main model
2. Create a binary target: `is_error = (y_pred != y_true)`
3. Train a secondary model to predict `is_error` using the same (or augmented) features
4. Interpret the secondary model: which features predict misclassification?

This reveals **systematic patterns** in your errors — subgroups where the model consistently fails.

### Practical Tip: Slice Before You Model

Before building a full contrast model, try simple error slicing:
- Group errors by categorical features (occupation, education, etc.)
- Compare error rates across subgroups
- Often, the pattern is obvious without a meta-model

📚 [Chung et al. (2019), "Slice Finder: Automated Data Slicing for Model Validation"](https://research.google/pubs/pub47966/)

In [ ]:
# ── Step 1: Simple error slicing ──
# Use GBM baseline predictions on validation set
y_pred_val = gbm_baseline.predict(X_val_tr)
y_prob_val = gbm_baseline.predict_proba(X_val_tr)[:, 1]
is_error = (y_pred_val != y_val).astype(int)

print(f"Overall error rate: {is_error.mean():.4f} ({is_error.sum()} errors out of {len(is_error)})")
print(f"  False positives: {((y_pred_val == 1) & (y_val == 0)).sum()}")
print(f"  False negatives: {((y_pred_val == 0) & (y_val == 1)).sum()}")

# Slice by categorical features from the original data
print("\n--- Error Rate by Subgroup ---")
for col in categorical_cols[:5]:  # first 5 categorical columns
    if col in val_df.columns:
        slice_df = val_df.copy()
        slice_df["is_error"] = is_error
        slice_rates = (
            slice_df.groupby(col)["is_error"]
            .agg(["mean", "count"])
            .rename(columns={"mean": "error_rate", "count": "n"})
            .sort_values("error_rate", ascending=False)
        )
        # Only show groups with at least 20 observations
        slice_rates = slice_rates[slice_rates["n"] >= 20]
        if len(slice_rates) > 1:
            print(f"\n  {col}:")
            for idx, row in slice_rates.head(5).iterrows():
                print(f"    {str(idx):<30s} error_rate={row['error_rate']:.3f}  (n={int(row['n'])})")

In [ ]:
# ── Step 2: Contrast model — train a meta-model to predict errors ──
# Use the preprocessed features + confidence as inputs
X_meta = np.column_stack([X_val_tr, y_prob_val])  # add model confidence as a feature
meta_feature_names = list(feature_names_ohe) + ["model_confidence"]

# Train a shallow tree to predict errors (interpretable)
meta_model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=SEED)

# Use cross-validation since we're fitting on the same val set
meta_cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
meta_scores = cross_val_score(meta_model, X_meta, is_error, cv=meta_cv, scoring="roc_auc")
print(f"Contrast model CV AUC: {meta_scores.mean():.4f} ± {meta_scores.std():.4f}")

if meta_scores.mean() > 0.55:
    print("The contrast model can predict errors better than chance.")
    print("This means errors are SYSTEMATIC, not random.")
else:
    print("Errors appear relatively random — no strong systematic pattern detected.")

# Fit on all val data for interpretation
meta_model.fit(X_meta, is_error)

meta_importances = pd.DataFrame({
    "feature": meta_feature_names,
    "importance": meta_model.feature_importances_,
}).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
top_meta = meta_importances[meta_importances["importance"] > 0].head(15)
ax.barh(top_meta["feature"], top_meta["importance"], color="coral", alpha=0.7)
ax.set_xlabel("Feature Importance (contrast model)")
ax.set_title("What Drives Misclassification?")
ax.invert_yaxis()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n'model_confidence' ranking high means the model 'knows' it's uncertain on hard cases.")
print("Other features ranking high reveal systematic subgroup failures.")

In [ ]:
# ── Step 3: Visualize error patterns ──
# Confidence distribution for correct vs incorrect predictions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

correct_probs = y_prob_val[is_error == 0]
error_probs = y_prob_val[is_error == 1]

axes[0].hist(correct_probs, bins=30, alpha=0.6, color="steelblue", label="Correct", density=True, edgecolor="white")
axes[0].hist(error_probs, bins=30, alpha=0.6, color="coral", label="Errors", density=True, edgecolor="white")
axes[0].set_xlabel("Model Confidence (P(positive))")
axes[0].set_ylabel("Density")
axes[0].set_title("Confidence Distribution: Correct vs Errors")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Error rate by confidence bin
confidence_bins = pd.cut(y_prob_val, bins=10)
error_by_conf = pd.DataFrame({"confidence_bin": confidence_bins, "is_error": is_error})
error_rates = error_by_conf.groupby("confidence_bin", observed=True)["is_error"].agg(["mean", "count"])
error_rates = error_rates[error_rates["count"] >= 5]

axes[1].bar(range(len(error_rates)), error_rates["mean"], color="coral", alpha=0.7, edgecolor="black")
axes[1].set_xticks(range(len(error_rates)))
axes[1].set_xticklabels([str(x) for x in error_rates.index], rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("Error Rate")
axes[1].set_title("Error Rate by Confidence Bin")
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print("Errors concentrate in the 'uncertain' region (0.3-0.7 confidence).")
print("This is expected — but checking which FEATURES are associated with uncertainty is the insight.")

> **Do It Yourself**
>
> Build a contrast model that distinguishes **false positives** from **true negatives** only
> (i.e., among the negative-class samples, what drives the model to predict positive?).
> This is often more actionable than modeling all errors together.
>
> Also try: instead of a DecisionTree meta-model, use a Random Forest and inspect its
> permutation importance. Does it reveal different patterns?

## <a id="stacking"></a> Section 9 — Ensemble Stacking & Blending

### Why Top Kaggle Solutions Almost Always Stack

Individual models have characteristic blind spots: a GBM might capture non-linear interactions well
but struggle with certain linear relationships, while a logistic regression captures those linear
effects cleanly. **Stacking** (also called stacked generalization) combines multiple diverse models
by training a **meta-learner** on their out-of-fold predictions.

### The Three Main Approaches

| Method | How It Works | Pros | Cons |
|--------|-------------|------|------|
| **Simple Averaging** | Average predicted probabilities | No overfitting risk, dead simple | Treats all models equally |
| **Weighted Averaging** | Weighted average (weights from CV) | Slightly better than equal weights | Need to tune weights |
| **Stacking** | Train a meta-model on OOF predictions | Can learn non-linear combinations | Risk of overfitting the meta-learner |

### The Critical Rule: Out-of-Fold Predictions

If you train base models on the full training set and then use their predictions as meta-features,
the meta-learner sees "cheating" inputs — predictions the base models made on data they trained on.
This leads to **meta-learner overfitting**.

The fix: generate predictions using **K-fold out-of-fold (OOF)** — each row's meta-feature comes
from a model that never saw that row during training. This is the same logic as cross-validation,
applied to prediction generation.

```
Fold 1: Train on folds 2-5 → predict fold 1
Fold 2: Train on folds 1,3-5 → predict fold 2
...
Result: every training row has an OOF prediction
```

📚 [Wolpert (1992), "Stacked Generalization"](https://www.sciencedirect.com/science/article/abs/pii/S0893608005800231) · [Kaggle Ensembling Guide](https://mlwave.com/kaggle-ensembling-guide/) · [sklearn StackingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html)

In [ ]:
# ── Step 1: Generate out-of-fold predictions from diverse base models ──
from sklearn.model_selection import cross_val_predict

skf_stack = StratifiedKFold(5, shuffle=True, random_state=SEED)

# Base models — diversity matters more than individual performance
base_models = {
    "LogisticRegression": LogisticRegression(C=1.0, max_iter=500, random_state=SEED),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, max_depth=12, max_features="sqrt",
        min_samples_leaf=10, random_state=SEED, n_jobs=-1,
    ),
    "GBM": GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=4,
        min_samples_leaf=20, random_state=SEED,
    ),
    "DecisionTree": DecisionTreeClassifier(
        max_depth=8, min_samples_leaf=20, random_state=SEED,
    ),
}

# Generate OOF predictions for each base model
# LogisticRegression needs scaled features; tree models use unscaled
oof_predictions = {}
val_predictions = {}
test_predictions = {}

for name, model in base_models.items():
    X_tr = X_train_sc if "Logistic" in name else X_train_tr
    X_v = X_val_sc if "Logistic" in name else X_val_tr
    X_te = X_test_sc if "Logistic" in name else X_test_tr

    # OOF predictions on training set (each row predicted by a model that didn't see it)
    oof_prob = cross_val_predict(model, X_tr, y_train, cv=skf_stack, method="predict_proba")[:, 1]
    oof_predictions[name] = oof_prob

    # Fit on full training set for val/test predictions
    model.fit(X_tr, y_train)
    val_predictions[name] = model.predict_proba(X_v)[:, 1]
    test_predictions[name] = model.predict_proba(X_te)[:, 1]

    oof_auc = roc_auc_score(y_train, oof_prob)
    val_auc = roc_auc_score(y_val, val_predictions[name])
    print(f"{name:<25s} OOF AUC: {oof_auc:.4f}   Val AUC: {val_auc:.4f}")

In [ ]:
# ── Step 2: Simple averaging vs weighted averaging ──

# Simple average
avg_val_prob = np.mean(list(val_predictions.values()), axis=0)
avg_val_auc = roc_auc_score(y_val, avg_val_prob)

# Weighted average — use OOF AUC as a proxy for weight
oof_aucs = {name: roc_auc_score(y_train, oof_predictions[name]) for name in base_models}
weights = np.array([oof_aucs[name] for name in base_models])
weights = weights / weights.sum()  # normalize

weighted_val_prob = np.average(list(val_predictions.values()), axis=0, weights=weights)
weighted_val_auc = roc_auc_score(y_val, weighted_val_prob)

print(f"{'Method':<30s} {'Val AUC':>10}")
print("-" * 42)
for name in base_models:
    auc = roc_auc_score(y_val, val_predictions[name])
    print(f"{name:<30s} {auc:>10.4f}")
print("-" * 42)
print(f"{'Simple Average':<30s} {avg_val_auc:>10.4f}")
print(f"{'Weighted Average (by OOF AUC)':<30s} {weighted_val_auc:>10.4f}")

print(f"\nWeights: {dict(zip(base_models.keys(), [f'{w:.3f}' for w in weights]))}")
print("\nEven simple averaging often beats the best individual model.")

In [ ]:
# ── Step 3: Stacking with a meta-learner ──
# Build meta-feature matrices from OOF predictions
X_meta_train = np.column_stack([oof_predictions[name] for name in base_models])
X_meta_val = np.column_stack([val_predictions[name] for name in base_models])
X_meta_test = np.column_stack([test_predictions[name] for name in base_models])

print(f"Meta-feature matrix shape: {X_meta_train.shape} (one column per base model)")

# Meta-learner: logistic regression on the base model predictions
# Keep it simple to avoid overfitting the meta-layer
meta_learner = LogisticRegression(C=1.0, max_iter=500, random_state=SEED)
meta_learner.fit(X_meta_train, y_train)

stacked_val_prob = meta_learner.predict_proba(X_meta_val)[:, 1]
stacked_val_auc = roc_auc_score(y_val, stacked_val_prob)
stacked_test_prob = meta_learner.predict_proba(X_meta_test)[:, 1]
stacked_test_auc = roc_auc_score(y_test, stacked_test_prob)

print(f"\nStacked ensemble Val AUC:  {stacked_val_auc:.4f}")
print(f"Stacked ensemble Test AUC: {stacked_test_auc:.4f}")

# Meta-learner coefficients — how much does it trust each base model?
meta_coefs = pd.DataFrame({
    "base_model": list(base_models.keys()),
    "meta_coefficient": meta_learner.coef_[0],
}).sort_values("meta_coefficient", ascending=False)
print(f"\nMeta-learner coefficients (how much weight each base model gets):")
for _, row in meta_coefs.iterrows():
    print(f"  {row['base_model']:<25s} {row['meta_coefficient']:+.3f}")

In [ ]:
# ── Step 4: sklearn's StackingClassifier (production-friendly shortcut) ──
from sklearn.ensemble import StackingClassifier

stacking_clf = StackingClassifier(
    estimators=[
        ("lr", LogisticRegression(C=1.0, max_iter=500, random_state=SEED)),
        ("rf", RandomForestClassifier(
            n_estimators=300, max_depth=12, max_features="sqrt",
            min_samples_leaf=10, random_state=SEED, n_jobs=-1,
        )),
        ("gbm", GradientBoostingClassifier(
            n_estimators=200, learning_rate=0.1, max_depth=4,
            min_samples_leaf=20, random_state=SEED,
        )),
    ],
    final_estimator=LogisticRegression(C=1.0, max_iter=500, random_state=SEED),
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    stack_method="predict_proba",
    n_jobs=-1,
)

# Note: StackingClassifier handles the OOF prediction logic internally
# It needs the scaled features for the LR base model — here we use tree features
# for simplicity (LR will still work, just slightly suboptimal)
print("Fitting StackingClassifier (handles OOF logic internally)...")
t0 = time.time()
stacking_clf.fit(X_train_tr, y_train)
elapsed = time.time() - t0

sklearn_stack_val = roc_auc_score(y_val, stacking_clf.predict_proba(X_val_tr)[:, 1])
sklearn_stack_test = roc_auc_score(y_test, stacking_clf.predict_proba(X_test_tr)[:, 1])

print(f"Done in {elapsed:.1f}s")
print(f"\nsklearn StackingClassifier Val AUC:  {sklearn_stack_val:.4f}")
print(f"sklearn StackingClassifier Test AUC: {sklearn_stack_test:.4f}")
print("\nStackingClassifier is convenient but less flexible than the manual approach —")
print("e.g., you can't easily use different preprocessing per base model.")

In [ ]:
# ── Summary: all ensemble methods compared ──
best_individual_name = max(val_predictions, key=lambda n: roc_auc_score(y_val, val_predictions[n]))
best_individual_auc = roc_auc_score(y_val, val_predictions[best_individual_name])

print(f"{'Method':<35s} {'Val AUC':>10}")
print("=" * 47)
for name in base_models:
    auc = roc_auc_score(y_val, val_predictions[name])
    marker = " ← best individual" if name == best_individual_name else ""
    print(f"  {name:<33s} {auc:>10.4f}{marker}")
print("-" * 47)
print(f"  {'Simple Average':<33s} {avg_val_auc:>10.4f}")
print(f"  {'Weighted Average':<33s} {weighted_val_auc:>10.4f}")
print(f"  {'Stacking (manual OOF)':<33s} {stacked_val_auc:>10.4f}")
print(f"  {'Stacking (sklearn)':<33s} {sklearn_stack_val:>10.4f}")
print("=" * 47)

print("\nKey takeaways:")
print("  1. Even simple averaging usually helps — it's the easiest win in competitive ML")
print("  2. Stacking adds complexity; the gain over averaging is often small")
print("  3. Diversity matters: stacking 5 GBMs is less useful than stacking GBM + LR + RF")
print("  4. In production, simple averaging is often preferred (easier to maintain and debug)")

> **Do It Yourself**
>
> Add a 5th base model (e.g., a second GBM with different hyperparameters, or an `ExtraTreesClassifier`)
> to the stack. Does the ensemble improve? What if you add a *bad* model — does averaging degrade?
>
> Advanced: try a **two-level stack** where the first level has 4 base models and the second level
> stacks their OOF predictions with the original features (meta-features + raw features).
> This is a common pattern in top Kaggle solutions.

---

## References

### Cross-Validation & Evaluation
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.). Springer. §7.10.
- Saito, T. & Rehmsmeier, M. (2015). The Precision-Recall Plot Is More Informative than the ROC Plot. *PLOS ONE*.
- Bergstra, J. & Bengio, Y. (2012). Random Search for Hyper-Parameter Optimization. *JMLR*, 13, 281–305.

### Dataset Shift & Adversarial Validation
- Quinonero-Candela, J. et al. (2009). *Dataset Shift in Machine Learning*. MIT Press.
- Pan, S. J. & Yang, Q. (2010). A Survey on Transfer Learning. *IEEE TKDE*, 22(10), 1345–1359.

### Tabular Foundation Models
- Hollmann, N. et al. (2023). TabPFN: A Transformer That Solves Small Tabular Classification Problems in a Second. *ICLR*.
- Ye, J. et al. (2025). TabICL: A Tabular In-Context Learning Approach. *arXiv:2501.02349*.

### Explainability
- Lundberg, S. M. & Lee, S. I. (2017). A Unified Approach to Interpreting Model Predictions. *NeurIPS*.
- Lundberg, S. M. et al. (2020). From Local Explanations to Global Understanding with Explainable AI for Trees. *Nature Machine Intelligence*.

### Hyperparameter Optimization
- Akiba, T. et al. (2019). Optuna: A Next-generation Hyperparameter Optimization Framework. *KDD*.

### Error Analysis
- Chung, Y. et al. (2019). Slice Finder: Automated Data Slicing for Model Validation. *ICDE*.

### Ensemble Stacking & Blending
- Wolpert, D. H. (1992). Stacked Generalization. *Neural Networks*, 5(2), 241–259.
- Breiman, L. (1996). Stacked Regressions. *Machine Learning*, 24(1), 49–64.
- Van der Laan, M. J. et al. (2007). Super Learner. *Statistical Applications in Genetics and Molecular Biology*, 6(1).